# EECS E6893 HW2 – Section 1: Streaming Analytics
**Workflow**
1. Every 5 minutes (7 pulls over 30 minutes) call the Massive aggregates API for AAPL at 1-minute resolution.
2. Because the free tier blocks same-day data, a *simulated clock* on a past trading day is used: each pull asks for `[sim_now - 1h, sim_now]`, and `sim_now` moves forward 5 minutes per pull.
3. Each pull is merged into `stock_df` incrementally: rows already present whose timestamp appears in the new pull are dropped (left **anti-join**) and replaced by the new rows; new timestamps are simply appended.
4. After each merge, 30-minute moving averages of c/l/h/o/v are computed and merged the same way into `ma_df`.

In [ ]:
!pip install pandas requests

In [1]:
import time
import requests
from datetime import datetime, timedelta, timezone

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

spark = SparkSession.builder.master("local[2]").appName("hw2-streaming").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")   # show all timestamps in UTC


API_KEY  = "o79slBorib8cw9sL2F796ZmOWLAPIl9h"         
BASE_URL = "https://api.massive.com"
TICKER   = "AAPL"

SIM_START = datetime(2026, 10, 1, 15, 0, tzinfo=timezone.utc)

N_PULLS      = 7      # t = 0, 5, 10, 15, 20, 25, 30 minutes
INTERVAL_SEC = 300    
MA_WINDOW_MIN = 30


26/10/08 17:32:01 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
raw_schema = StructType([
    StructField("Stock Name", StringType()),
    StructField("t", LongType()),
    StructField("c", DoubleType()),
    StructField("l", DoubleType()),
    StructField("h", DoubleType()),
    StructField("o", DoubleType()),
    StructField("v", DoubleType()),
])

def get_histdata_massive(ticker, start_dt, end_dt, timespan="minute", multiplier=1):
    """Pull 1-minute bars between start_dt and end_dt (UTC datetimes) and return a Spark DataFrame
    with schema ['Stock Name', 'UTC Timestamp', 'c', 'l', 'h', 'o', 'v']."""
    start_ms = int(start_dt.timestamp() * 1000)
    end_ms   = int(end_dt.timestamp() * 1000)
    url = f"{BASE_URL}/v2/aggs/ticker/{ticker}/range/{multiplier}/{timespan}/{start_ms}/{end_ms}"
    resp = requests.get(url, params={"adjusted": "true", "sort": "asc",
                                     "limit": 50000, "apiKey": API_KEY}, timeout=30)
    results = []
    if resp.status_code == 200:
        results = resp.json().get("results", []) or []
        if not results:
            print("No results returned for the specified window.")
    elif resp.status_code == 403:
        print("Error 403: Forbidden. Check your API key / use a past trading day.")
    else:
        print(f"Error {resp.status_code}: {resp.text}")

    rows = [(ticker, int(r["t"]), float(r["c"]), float(r["l"]), float(r["h"]),
             float(r["o"]), float(r["v"])) for r in results]
    df = spark.createDataFrame(rows, raw_schema)
    # convert the UNIX millisecond timestamp under key "t" into a UTC timestamp
    df = df.withColumn("UTC Timestamp", F.timestamp_seconds(F.col("t") / 1000))
    return df.select("Stock Name", "UTC Timestamp", "c", "l", "h", "o", "v")


def incremental_update(existing_df, new_df, keys):
    """Left anti-join: keep only old rows whose key is NOT in the new pull,
    then append the whole new pull (overlapping rows are replaced, new rows are added)."""
    if existing_df is None:
        merged = new_df
    else:
        kept = existing_df.join(new_df.select(*keys), on=keys, how="left_anti")
        merged = kept.unionByName(new_df)
    return merged.localCheckpoint()   # materialize & cut lineage so the plan doesn't grow every loop


def compute_moving_averages(df):
    """30-minute moving average for every timestamp; Datetime = end of the MA window."""
    secs = F.col("UTC Timestamp").cast("long")
    w = (Window.partitionBy("Stock Name").orderBy(secs)
               .rangeBetween(-(MA_WINDOW_MIN * 60 - 1), 0))   # the 30 one-minute bars ending at this row
    return df.select(
        F.col("UTC Timestamp").alias("Datetime"),
        F.avg("c").over(w).alias("c_MA"),
        F.avg("l").over(w).alias("l_MA"),
        F.avg("h").over(w).alias("h_MA"),
        F.avg("o").over(w).alias("o_MA"),
        F.avg("v").over(w).alias("v_MA"),
    )

# quick test of a single pull
get_histdata_massive(TICKER, SIM_START - timedelta(hours=1), SIM_START).show(5, truncate=False)

+----------+-------------------+------+------+--------+--------+-------------+
|Stock Name|UTC Timestamp      |c     |l     |h       |o       |v            |
+----------+-------------------+------+------+--------+--------+-------------+
|AAPL      |2026-10-01 14:00:00|329.83|329.81|330.555 |330.39  |191744.629933|
|AAPL      |2026-10-01 14:01:00|329.39|329.34|329.8499|329.8499|131280.36942 |
|AAPL      |2026-10-01 14:02:00|329.03|328.83|329.41  |329.37  |133898.080529|
|AAPL      |2026-10-01 14:03:00|329.3 |328.89|329.35  |329.01  |96841.84943  |
|AAPL      |2026-10-01 14:04:00|328.89|328.63|329.3499|329.29  |140232.057392|
+----------+-------------------+------+------+--------+--------+-------------+
only showing top 5 rows



In [3]:
stock_df = None
ma_df = None

for i in range(N_PULLS):
    loop_start = time.time()
    sim_now   = SIM_START + timedelta(minutes=5 * i)
    win_start = sim_now - timedelta(hours=1)
    print("=" * 90)
    print(f"Pull {i+1}/{N_PULLS} | real time {datetime.now(timezone.utc):%H:%M:%S} UTC | "
          f"window {win_start:%Y-%m-%d %H:%M} -> {sim_now:%H:%M} UTC")

    # (1) pull one hour of 1-minute data
    new_df = get_histdata_massive(TICKER, win_start, sim_now)
    print(f"rows in this pull: {new_df.count()}")

    # (2) incremental load into the stock dataframe
    stock_df = incremental_update(stock_df, new_df, ["Stock Name", "UTC Timestamp"])
    print(f"rows in stock_df after update: {stock_df.count()}")
    stock_df.orderBy("UTC Timestamp").show(1000, truncate=False)

    # (3) recompute 30-min moving averages and store them incrementally
    ma_new = compute_moving_averages(stock_df)
    ma_df = incremental_update(ma_df, ma_new, ["Datetime"])
    print(f"rows in ma_df after update: {ma_df.count()}")
    ma_df.orderBy("Datetime").show(1000, truncate=False)

    if i < N_PULLS - 1:
        time.sleep(max(0, INTERVAL_SEC - (time.time() - loop_start)))

print("Done: API called", N_PULLS, "times.")

Pull 1/7 | real time 17:35:58 UTC | window 2026-10-01 14:00 -> 15:00 UTC


rows in this pull: 61


rows in stock_df after update: 61
+----------+-------------------+--------+--------+--------+--------+-------------+
|Stock Name|UTC Timestamp      |c       |l       |h       |o       |v            |
+----------+-------------------+--------+--------+--------+--------+-------------+
|AAPL      |2026-10-01 14:00:00|329.83  |329.81  |330.555 |330.39  |191744.629933|
|AAPL      |2026-10-01 14:01:00|329.39  |329.34  |329.8499|329.8499|131280.36942 |
|AAPL      |2026-10-01 14:02:00|329.03  |328.83  |329.41  |329.37  |133898.080529|
|AAPL      |2026-10-01 14:03:00|329.3   |328.89  |329.35  |329.01  |96841.84943  |
|AAPL      |2026-10-01 14:04:00|328.89  |328.63  |329.3499|329.29  |140232.057392|
|AAPL      |2026-10-01 14:05:00|329.315 |328.88  |329.38  |328.89  |112206.173566|
|AAPL      |2026-10-01 14:06:00|329.4   |329.06  |329.52  |329.29  |87908.147521 |
|AAPL      |2026-10-01 14:07:00|329.11  |329.03  |329.46  |329.41  |103706.613295|
|AAPL      |2026-10-01 14:08:00|329.595 |329.08  |329

rows in ma_df after update: 61
+-------------------+------------------+------------------+------------------+------------------+------------------+
|Datetime           |c_MA              |l_MA              |h_MA              |o_MA              |v_MA              |
+-------------------+------------------+------------------+------------------+------------------+------------------+
|2026-10-01 14:00:00|329.83            |329.81            |330.555           |330.39            |191744.629933     |
|2026-10-01 14:01:00|329.61            |329.575           |330.20245         |330.11995         |161512.49967649998|
|2026-10-01 14:02:00|329.4166666666667 |329.32666666666665|329.9383          |329.8699666666667 |152307.693294     |
|2026-10-01 14:03:00|329.3875          |329.2175          |329.79122500000005|329.65497500000004|138441.232328     |
|2026-10-01 14:04:00|329.288           |329.1             |329.70296         |329.58198000000004|138799.3973408    |
|2026-10-01 14:05:00|329.2925    

In [4]:
# Final results
print("Final streamed data:", stock_df.count(), "rows")
stock_df.agg(F.min("UTC Timestamp").alias("first"), F.max("UTC Timestamp").alias("last")).show(truncate=False)
stock_df.orderBy("UTC Timestamp").show(1000, truncate=False)

print("Final moving averages:", ma_df.count(), "rows")
ma_df.orderBy("Datetime").show(1000, truncate=False)

Final streamed data: 91 rows
+-------------------+-------------------+
|first              |last               |
+-------------------+-------------------+
|2026-10-01 14:00:00|2026-10-01 15:30:00|
+-------------------+-------------------+

+----------+-------------------+--------+--------+--------+--------+-------------+
|Stock Name|UTC Timestamp      |c       |l       |h       |o       |v            |
+----------+-------------------+--------+--------+--------+--------+-------------+
|AAPL      |2026-10-01 14:00:00|329.83  |329.81  |330.555 |330.39  |191744.629933|
|AAPL      |2026-10-01 14:01:00|329.39  |329.34  |329.8499|329.8499|131280.36942 |
|AAPL      |2026-10-01 14:02:00|329.03  |328.83  |329.41  |329.37  |133898.080529|
|AAPL      |2026-10-01 14:03:00|329.3   |328.89  |329.35  |329.01  |96841.84943  |
|AAPL      |2026-10-01 14:04:00|328.89  |328.63  |329.3499|329.29  |140232.057392|
|AAPL      |2026-10-01 14:05:00|329.315 |328.88  |329.38  |328.89  |112206.173566|
|AAPL      |2